# ML & AI NEXUS 2026 - Final Winning Solution
**Team Name:** codewave  
**Members:** Sahil Kavishka. sarasi bimsara, pooja anabalagan and Aradhya jayawardhana 
**Theme:** Beyond the Black Box: Statistics for Trustworthy AI  
**Selected Submission:** codewave submission 9.csv / codewave submission 12.csv  

This reproducible notebook trains our multi-model ensemble (CatBoost + LightGBM + Calibrated Lasso Logistic Regression) with clinical domain feature engineering, covariate-shift adaptation, and probability calibration.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from catboost import CatBoostClassifier
import lightgbm as lgb

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

train_raw = pd.read_csv('train.csv')
test_raw = pd.read_csv('test.csv')
print(f'Train shape: {train_raw.shape}, Test shape: {test_raw.shape}')

## 1. Feature Engineering & Clinical Domain Risk Indicators

In [ ]:
def feature_engineer(df):
    df = df.copy()
    df['anemia_flag'] = (df['hemoglobin_g_dl'] < 11.0).astype(float)
    df['aki_flag'] = (df['creatinine_mg_dl'] > 1.5).astype(float)
    df['hyponatremia_flag'] = (df['sodium_mmol_l'] < 135).astype(float)
    df['tachy_flag'] = (df['heart_rate_bpm'] > 100).astype(float)
    df['htn_bp_flag'] = (df['systolic_bp_mmhg'] > 140).astype(float)
    for c in ['hemoglobin_g_dl', 'creatinine_mg_dl', 'sodium_mmol_l', 'heart_rate_bpm', 'systolic_bp_mmhg', 'followup_days']:
        df[c + '_missing'] = df[c].isnull().astype(int)
    df['comorbidity_load'] = df['diabetes'] + df['hypertension'] + df['chronic_kidney_disease'] + df['heart_failure']
    df['cardiorenal'] = (df['chronic_kidney_disease'] & df['heart_failure']).astype(int)
    df['polypharmacy'] = (df['medication_count'] >= 10).astype(int)
    df['frequent_admitter'] = (df['prior_admissions_12m'] >= 2).astype(int)
    df['long_stay'] = (df['length_of_stay_days'] >= 7).astype(int)
    df['late_followup'] = ((df['followup_days'] > 14) | df['followup_days'].isnull()).astype(int)
    df['simple_risk_score'] = df['frequent_admitter'] + df['long_stay'] + df['polypharmacy'] + df['anemia_flag'] + df['aki_flag']
    return df

train_fe = feature_engineer(train_raw)
test_fe = feature_engineer(test_raw)
print('Feature engineering complete!')

## 2. Model Training & Out-of-Fold Cross-Validation

In [ ]:
CAT_COLS = ['sex', 'rurality', 'hospital_type', 'region', 'discharge_disposition', 'care_pathway']
NUM_COLS = [c for c in train_fe.columns if c not in CAT_COLS + ['patient_id', 'readmitted_30d']]
FEATURES = NUM_COLS + CAT_COLS
y = train_fe['readmitted_30d']

# Native categories for CatBoost and LightGBM
X_native = train_fe[FEATURES].copy()
X_test_native = test_fe[FEATURES].copy()
for c in CAT_COLS:
    X_native[c] = X_native[c].astype('category')
    X_test_native[c] = X_test_native[c].astype('category')

# Preprocessing for linear models
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
train_ohe = ohe.fit_transform(train_fe[CAT_COLS].astype(str))
test_ohe = ohe.transform(test_fe[CAT_COLS].astype(str))
imp = IterativeImputer(random_state=SEED, max_iter=8, n_nearest_features=6)
train_num_imp = imp.fit_transform(train_fe[NUM_COLS])
test_num_imp = imp.transform(test_fe[NUM_COLS])
scaler = StandardScaler()
Xs = np.hstack([scaler.fit_transform(train_num_imp), train_ohe])
Xs_t = np.hstack([scaler.transform(test_num_imp), test_ohe])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print('Data prepared for 5-fold CV.')

## 3. Training Stacking Ensemble (CatBoost + LightGBM + Lasso-LR)

In [ ]:
cb_oof = np.zeros(len(X_native)); cb_test = np.zeros(len(X_test_native))
lgb_oof = np.zeros(len(X_native)); lgb_test = np.zeros(len(X_test_native))
lr_oof = np.zeros(len(Xs)); lr_test = np.zeros(len(Xs_t))

for ti, vi in skf.split(X_native, y):
    # CatBoost
    cb = CatBoostClassifier(iterations=1500, learning_rate=0.03, depth=5, cat_features=CAT_COLS, verbose=False, random_seed=SEED)
    cb.fit(X_native.iloc[ti], y.iloc[ti], eval_set=(X_native.iloc[vi], y.iloc[vi]), early_stopping_rounds=80)
    cb_oof[vi] = cb.predict_proba(X_native.iloc[vi])[:, 1]
    cb_test += cb.predict_proba(X_test_native)[:, 1] / 5.0
    
    # LightGBM
    lg = lgb.LGBMClassifier(learning_rate=0.025, num_leaves=24, max_depth=4, random_state=SEED, verbose=-1, n_jobs=-1)
    lg.fit(X_native.iloc[ti], y.iloc[ti], eval_set=[(X_native.iloc[vi], y.iloc[vi])], callbacks=[lgb.early_stopping(80, verbose=False)])
    lgb_oof[vi] = lg.predict_proba(X_native.iloc[vi])[:, 1]
    lgb_test += lg.predict_proba(X_test_native)[:, 1] / 5.0
    
    # Calibrated Lasso LR
    base_lr = LogisticRegression(C=0.15, penalty='l1', solver='saga', max_iter=1500, random_state=SEED)
    m = CalibratedClassifierCV(base_lr, method='sigmoid', cv=3)
    m.fit(Xs[ti], y.iloc[ti])
    lr_oof[vi] = m.predict_proba(Xs[vi])[:, 1]
    lr_test += m.predict_proba(Xs_t)[:, 1] / 5.0

print(f'CatBoost OOF LogLoss: {log_loss(y, cb_oof):.5f} | AUC: {roc_auc_score(y, cb_oof):.5f}')
print(f'LightGBM OOF LogLoss: {log_loss(y, lgb_oof):.5f} | AUC: {roc_auc_score(y, lgb_oof):.5f}')
print(f'Lasso-LR OOF LogLoss: {log_loss(y, lr_oof):.5f} | AUC: {roc_auc_score(y, lr_oof):.5f}')

## 4. Final Submission Generation with Probability Bounds

In [ ]:
# Optimal Stacking Blend
final_test = 0.50 * lr_test + 0.30 * cb_test + 0.20 * lgb_test
final_probs = np.clip(final_test, 0.0248, 0.7280)

sub = pd.DataFrame({
    'patient_id': test_raw['patient_id'],
    'readmitted_30d': final_probs
})
sub.to_csv('codewave_final_submission.csv', index=False)
print('Saved codewave_final_submission.csv successfully!')
print(sub.head(10))